# Import required libraries

In [3]:
import operator
import functools
import pandas as pd
from PIL import Image
from pydantic import BaseModel

# typing decorators
from typing import List, Tuple, Dict, Any, Sequence, Annotated, Literal
from typing_extensions import TypedDict

# langchain packages
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import (StrOutputParser, 
                                            JsonOutputParser)
from langchain_core.prompts import ChatPromptTemplate


# langgraph packages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, create_react_agent

# Transformer packages
from transformers import (AutoModelForImageClassification, 
                            AutoProcessor)


# Load custom defined packages
from models import (GenderPredictionFromPreTrainedModel, 
                    AgePredictionForPretrainedModel)

In [12]:
# Load patient dataset
updated_patient_df = pd.read_csv('datasets/input/processed/updated_patient_df.csv')
updated_patient_df.drop('Unnamed: 0', axis = 1, inplace = True)
updated_patient_df.head(10)


# Pick a Random Patient: A female between 20 and 29 and with Pneumonia as Positive 
selected_patients = updated_patient_df[
    (updated_patient_df['Gender'] == 'Female') & \
        (updated_patient_df['Age'].between(20, 29)) & \
            (updated_patient_df['Difficulty Breathing'] == 'Yes') & \
                (updated_patient_df['Outcome Variable'] == 'Positive')]

# Choosing first patient details as our subject
selected_patients = (selected_patients
                                .reset_index(drop = True)
                                .iloc[0, :])

selected_patients


Disease                     Influenza
Fever                             Yes
Cough                             Yes
Fatigue                           Yes
Difficulty Breathing              Yes
Age                                25
Gender                         Female
Blood Pressure                 Normal
Cholesterol Level              Normal
Outcome Variable             Positive
First_Name                     Amrita
Last_Name                        Nair
Patient_ID              PAT1000000007
Full_Name                 Amrita_Nair
Name: 0, dtype: object